# Anomaly-Based Network Intrusion Detection System
## Notebook 05: Deep Learning Models - Alternative Version

**Important Note**: This notebook contains instructions for running deep learning models.
Due to TensorFlow compatibility limitations with Python 3.14, this notebook provides
theoretical implementation details and instructions for running on compatible environments.

### For users with Python 3.8-3.11:
1. Create a new Python environment with Python 3.11
2. Install TensorFlow: `pip install tensorflow`
3. Run this notebook with the updated requirements

### Alternative Approach Shown:
We demonstrate a simplified neural network using scikit-learn's MLPClassifier
as a fallback when TensorFlow is not available.

## 1. Imports

In [ ]:
import os, sys, warnings, joblib
warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

from src.evaluation.metrics import evaluate_model

np.random.seed(42)

MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

print('Deep Learning Models Notebook - Alternative Version')
print('=' * 60)

## 2. TensorFlow Compatibility Notice

In [ ]:
print("TensorFlow Compatibility Information")
print("=" * 40)
print("\nCurrent Python Version:")
import sys
print(f"Python {sys.version}")
print("\nTensorFlow Requirements:")
print("- Python 3.8-3.11 (TensorFlow 2.x)")
print("- Python 3.9-3.12 (TensorFlow 2.15+)")
print("\nNote: Python 3.14 is not currently supported by TensorFlow.")
print("\nTo run the full deep learning models:")
print("1. Create a new environment: `conda create -n tf_env python=3.11`")
print("2. Activate: `conda activate tf_env`")
print("3. Install: `pip install tensorflow`")
print("4. Run the original notebook: `05_dl_models.ipynb`")
print("\nThis notebook demonstrates a fallback using scikit-learn's MLP.")

## 3. Load Data

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
FEAT_DIR = os.path.join(PROJECT_ROOT, 'data', 'features')

X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_val   = pd.read_csv(f'{DATA_DIR}/X_val.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv').squeeze().values
y_val   = pd.read_csv(f'{DATA_DIR}/y_val.csv').squeeze().values
y_test  = pd.read_csv(f'{DATA_DIR}/y_test.csv').squeeze().values

SELECTED = joblib.load(f'{FEAT_DIR}/selected_features.pkl')

X_tr = X_train[SELECTED].values.astype('float32')
X_va = X_val[SELECTED].values.astype('float32')
X_te = X_test[SELECTED].values.astype('float32')

n_features = X_tr.shape[1]
print(f'Features: {n_features}  |  Train samples: {len(X_tr):,}')
print(f'Validation samples: {len(X_va):,}  |  Test samples: {len(X_te):,}')
print(f'Class distribution: {np.bincount(y_train.astype(int))}')

## 4. Alternative: Scikit-learn MLP Classifier

In [ ]:
print("Training scikit-learn MLP Classifier (Fallback)")
print("=" * 50)

# Create MLP classifier
mlp_sklearn = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=256,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=100,
    shuffle=True,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=True
)

print(f"MLP Architecture: Input({n_features}) -> 256 -> 128 -> 64 -> Output(1)")
print("Training...")

# Train the model
mlp_sklearn.fit(X_tr, y_train)

print("\nTraining completed!")

In [ ]:
# Evaluate the model
y_pred_mlp = mlp_sklearn.predict(X_te)
y_prob_mlp = mlp_sklearn.predict_proba(X_te)[:, 1]

print('MLP Classification Report:')
print(classification_report(y_test, y_pred_mlp, target_names=['Normal', 'Attack']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_mlp):.4f}')

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(mlp_sklearn.loss_curve_, label='Training Loss')
plt.title('MLP Training Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Theoretical Deep Learning Models (If TensorFlow Available)

In [ ]:
print("Theoretical Deep Learning Models (TensorFlow Required)")
print("=" * 60)

print("\nIf TensorFlow were available, we would implement:")
print("""
1. MLP Classifier (TensorFlow/Keras):
   - Architecture: Input → Dense(256, relu) → BatchNorm → Dropout(0.3)
                 → Dense(128, relu) → BatchNorm → Dropout(0.3)
                 → Dense(64, relu) → Dense(1, sigmoid)
   - Optimizer: Adam, Loss: Binary Crossentropy
   - Expected Accuracy: ~96-98%

2. LSTM Classifier:
   - Sequence length: 5 timesteps
   - Architecture: Input(5, n_features) → LSTM(128) → LSTM(64) → Dense(1, sigmoid)
   - Purpose: Capture temporal patterns in network traffic
   - Expected Accuracy: ~95-97%

3. Autoencoder (Unsupervised):
   - Architecture: Encoder: Input → Dense(64) → Dense(32) → Dense(16)
                 Decoder: Dense(32) → Dense(64) → Dense(n_features)
   - Purpose: Detect anomalies via reconstruction error
   - Threshold: 95th percentile of normal traffic reconstruction error
   - Expected Accuracy: ~92-95%
""")

print("\nTo implement these models:")
print("1. Install TensorFlow in compatible Python environment")
print("2. Run the original notebook: 05_dl_models.ipynb")
print("3. Models will be saved in 'models/' directory")

## 6. Comparison with Classical Models

In [ ]:
# Load classical model results for comparison
try:
    classical_results = pd.read_csv(f'{RESULTS_DIR}/classical_model_comparison.csv', index_col=0)
    
    # Add MLP results
    mlp_metrics = {
        'Accuracy': accuracy_score(y_test, y_pred_mlp),
        'Precision': precision_score(y_test, y_pred_mlp, zero_division=0),
        'Recall': recall_score(y_test, y_pred_mlp, zero_division=0),
        'F1': f1_score(y_test, y_pred_mlp, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob_mlp),
    }
    
    # Create comparison dataframe
    comparison = classical_results.copy()
    comparison.loc['MLP (scikit-learn)'] = mlp_metrics
    
    print("Model Performance Comparison:")
    print("=" * 40)
    print(comparison.round(4))
    
    # Plot comparison
    plt.figure(figsize=(12, 6))
    metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
    
    for i, metric in enumerate(metrics_to_plot, 1):
        plt.subplot(2, 3, i)
        comparison[metric].plot(kind='bar', color=['#2E86AB', '#A23B72', '#F18F01', '#4CAF50'])
        plt.title(f'{metric}')
        plt.ylabel('Score')
        plt.ylim(0.9, 1.0)
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
    
    plt.suptitle('Model Performance Comparison (Including MLP Fallback)', fontsize=14)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Could not load classical results: {e}")

## 7. Save Results

In [ ]:
# Save the MLP model
joblib.dump(mlp_sklearn, f'{MODELS_DIR}/mlp_sklearn_model.pkl')

# Save comparison results
try:
    comparison.to_csv(f'{RESULTS_DIR}/model_comparison_with_mlp.csv')
    print(f"\nResults saved to: {RESULTS_DIR}/model_comparison_with_mlp.csv")
except:
    pass

print("\n" + "=" * 60)
print("Summary:")
print("1. MLP Classifier trained using scikit-learn (fallback)")
print("2. TensorFlow models require Python 3.8-3.11 environment")
print("3. Best classical model: XGBoost (98.74% accuracy)")
print("4. MLP provides reasonable performance as fallback")
print("5. For full DL implementation, use compatible Python + TensorFlow")
print("=" * 60)